# 01 — Setup

This notebook prepares the environment for the TCGA-LUAD workflow.

It uses:
- `requirements.txt` for Python packages,
- the GRCh38 reference download for SigProfiler,
- simple folder checks before the analysis starts.


## 1. Install Python packages

Run this cell once.  
If the notebook still complains about missing packages after installation, restart the kernel and continue.


In [ ]:
%pip install -r requirements.txt


## 2. Download the GRCh38 reference for SigProfiler

This is the only extra reference download needed for the pipeline.


In [ ]:
from SigProfilerMatrixGenerator import install as gen_install

gen_install.install("GRCh38", rsync=False, bash=True)


## 3. Optional Colab step

Use this only if you are working in Google Colab and uploaded a ZIP file with the project data.


In [ ]:
# !unzip -q data.zip
# !ls -la
# !find data -maxdepth 2 -type d | sort


## 4. Check the working directory and folders


In [ ]:
%matplotlib inline

from pathlib import Path
import re

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import Image, display, Markdown

PROJECT_ROOT = Path.cwd().resolve()

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)

def show_image(path, width=1000):
    path = Path(path)
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f"Image not found: {path}")

def show_images(*paths, width=1000):
    for path in paths:
        show_image(path, width=width)

def get_sbs_columns(columns):
    cols = [col for col in columns if str(col).startswith("SBS") and str(col)[3:].isdigit()]
    return sorted(cols, key=lambda col: int(str(col)[3:]))

def normalize_rows(frame):
    frame = frame.copy()
    frame = frame.apply(pd.to_numeric, errors="coerce").fillna(0.0)
    row_sums = frame.sum(axis=1)
    frame = frame.loc[row_sums > 0].copy()
    return frame.div(frame.sum(axis=1), axis=0)

def extract_patient_id(value):
    match = re.search(r"(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})", str(value))
    return match.group(1) if match else np.nan

def map_smoking_label(value):
    if pd.isna(value):
        return "Unknown"

    text = str(value).strip().lower()

    if text in {"", "nan", "not reported", "unknown"}:
        return "Unknown"
    if "never" in text or "lifelong" in text or "non-smoker" in text or "nonsmoker" in text:
        return "Never"
    if "current reformed" in text or "reformed" in text or "former" in text:
        return "Former"
    if "current" in text:
        return "Current"
    return "Unknown"

def set_spaced_xticks(ax, labels, step=3):
    ax.set_xticks(np.arange(len(labels)) + 0.5)
    ax.set_xticklabels(labels, rotation=90)
    for i, label in enumerate(ax.get_xticklabels()):
        if i % step != 0:
            label.set_visible(False)


    expected_paths = {
        "data": PROJECT_ROOT / "data",
        "maf": PROJECT_ROOT / "data" / "maf",
        "results": PROJECT_ROOT / "results",
        "plots": PROJECT_ROOT / "plots",
    }

    print("Project root:", PROJECT_ROOT)
    print()

    status_rows = []
    for name, path in expected_paths.items():
        status_rows.append(
            {
                "name": name,
                "path": str(path),
                "exists": path.exists(),
            }
        )

    status_df = pd.DataFrame(status_rows)
    display(status_df)


## 5. Quick note

Recommended notebook order:
1. `01_setup_environment.ipynb`
2. `02_sigprofiler_generation_and_fitting.ipynb`
3. `03_merge_clinical_and_exposure.ipynb`
4. `04_unsupervised_signature_plots.ipynb`
5. `05_supervised_data_preparation.ipynb`
6. `06_supervised_balanced_random_forest.ipynb`
